# Fine-tuning Large Language Models on HPC

This notebook covers fine-tuning techniques for Large Language Models on HPC clusters. We'll explore efficient fine-tuning methods like LoRA, QLoRA, and distributed training.

## Learning Objectives

* Understand fine-tuning concepts and techniques
* Implement LoRA (Low-Rank Adaptation) for efficient training
* Use QLoRA for memory-efficient fine-tuning
* Set up distributed training across multiple GPUs
* Evaluate and compare fine-tuned models
* Deploy fine-tuned models for inference

## Setup and Imports

Let's import the necessary libraries for fine-tuning.

In [ ]:
import torch
import transformers
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, 
    TrainingArguments, Trainer, DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig, get_peft_model, TaskType,
    PeftModel, PeftConfig
)
import datasets
import accelerate
import bitsandbytes
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Fine-tuning environment setup complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

## Fine-tuning Concepts

### What is Fine-tuning?

Fine-tuning adapts a pre-trained language model to specific tasks or domains by training on task-specific data while preserving the model's general knowledge.

### Fine-tuning Methods:

1. **Full Fine-tuning**: Train all model parameters
   - Pros: Maximum performance potential
   - Cons: Requires significant memory and compute

2. **LoRA (Low-Rank Adaptation)**: Train only low-rank matrices
   - Pros: Memory efficient, fast training
   - Cons: Slightly reduced performance

3. **QLoRA**: LoRA with quantization
   - Pros: Extremely memory efficient
   - Cons: More complex setup

4. **Adapter Layers**: Train small adapter modules
   - Pros: Modular, reusable
   - Cons: Additional complexity

## Preparing Training Data

Let's create a sample dataset for fine-tuning. We'll use a simple instruction-following format.

In [ ]:
def create_sample_dataset():
    """Create a sample instruction-following dataset"""
    
    # Sample training data for a coding assistant
    training_data = [
        {
            "instruction": "Write a Python function to calculate factorial",
            "input": "",
            "output": "def factorial(n):\n    if n <= 1:\n        return 1\n    return n * factorial(n - 1)"
        },
        {
            "instruction": "Explain what machine learning is",
            "input": "",
            "output": "Machine learning is a subset of artificial intelligence that enables computers to learn and make decisions from data without being explicitly programmed for every task."
        },
        {
            "instruction": "Write a SQL query to find the top 10 customers",
            "input": "Table: customers with columns: id, name, total_purchases",
            "output": "SELECT id, name, total_purchases\nFROM customers\nORDER BY total_purchases DESC\nLIMIT 10;"
        },
        {
            "instruction": "What are the benefits of using GPUs for AI?",
            "input": "",
            "output": "GPUs provide several benefits for AI: 1) Parallel processing capabilities for matrix operations, 2) High memory bandwidth, 3) Specialized tensor cores for deep learning, 4) Cost-effective scaling for training large models."
        },
        {
            "instruction": "Write a Python function to sort a list",
            "input": "",
            "output": "def sort_list(lst):\n    return sorted(lst)\n\n# Or using the list method:\ndef sort_list_inplace(lst):\n    lst.sort()\n    return lst"
        },
        {
            "instruction": "Explain the difference between supervised and unsupervised learning",
            "input": "",
            "output": "Supervised learning uses labeled training data to learn a mapping from inputs to outputs. Unsupervised learning finds patterns in data without labeled examples, such as clustering or dimensionality reduction."
        },
        {
            "instruction": "Write a regex pattern to match email addresses",
            "input": "",
            "output": "import re\n\nemail_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$'\n\ndef is_valid_email(email):\n    return re.match(email_pattern, email) is not None"
        },
        {
            "instruction": "What is the purpose of validation data in machine learning?",
            "input": "",
            "output": "Validation data is used to tune hyperparameters and assess model performance during training. It helps prevent overfitting and provides an unbiased evaluation of the model's generalization ability."
        }
    ]
    
    return training_data

# Create dataset
dataset = create_sample_dataset()
print(f"Created dataset with {len(dataset)} examples")

# Display sample
print("\nSample training example:")
sample = dataset[0]
print(f"Instruction: {sample['instruction']}")
print(f"Output: {sample['output']}")

## Formatting Data for Training

We need to format our data into the proper structure for fine-tuning.

In [ ]:
def format_instruction_data(example):
    """Format instruction data for training"""
    if example['input']:
        prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{example['instruction']}\n\n### Response:\n"
    
    response = example['output']
    
    # Combine prompt and response
    full_text = prompt + response
    
    return {
        "text": full_text,
        "prompt": prompt,
        "response": response
    }

# Format all training data
formatted_data = [format_instruction_data(example) for example in dataset]

# Convert to HuggingFace dataset
from datasets import Dataset
train_dataset = Dataset.from_list(formatted_data)

print(f"Formatted dataset: {len(train_dataset)} examples")
print("\nSample formatted text:")
print(train_dataset[0]['text'])
print("\n" + "="*50)

## Loading Base Model

Let's load a base model for fine-tuning. We'll use a smaller model that fits well on L4 GPUs.

In [ ]:
# Model configuration for fine-tuning
MODEL_NAME = "microsoft/phi-2"  # Good for L4 GPU
# Alternative: "mistralai/Mistral-7B-Instruct-v0.1" for larger models

print(f"Loading model: {MODEL_NAME}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Configure quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

print(f"Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Check GPU memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"GPU memory allocated: {allocated:.2f} GB")

## LoRA Configuration

Let's set up LoRA (Low-Rank Adaptation) for efficient fine-tuning.

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,  # Rank of adaptation
    lora_alpha=32,  # Scaling parameter
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

print("\nLoRA configuration:")
print(f"Rank (r): {lora_config.r}")
print(f"Alpha: {lora_config.lora_alpha}")
print(f"Target modules: {lora_config.target_modules}")
print(f"Dropout: {lora_config.lora_dropout}")

## Data Collator

We need a data collator to handle tokenization and padding for training.

In [ ]:
def tokenize_function(examples):
    """Tokenize the training examples"""
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=512,
        return_tensors=None,
    )

# Tokenize dataset
tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # We're doing causal language modeling, not masked
)

print(f"Tokenized dataset: {len(tokenized_dataset)} examples")
print(f"Sample tokenized length: {len(tokenized_dataset[0]['input_ids'])}")

# Split into train/validation
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset_final = split_dataset["train"]
eval_dataset_final = split_dataset["test"]

print(f"Training examples: {len(train_dataset_final)}")
print(f"Validation examples: {len(eval_dataset_final)}")

## Training Arguments

Configure the training parameters for our fine-tuning run.

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=1,  # Small batch for L4 GPU
    per_device_eval_batch_size=1,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,  # Use mixed precision
    logging_steps=1,
    evaluation_strategy="steps",
    eval_steps=2,
    save_steps=10,
    save_total_limit=2,
    remove_unused_columns=False,
    push_to_hub=False,
    report_to=None,  # Disable wandb/tensorboard
    dataloader_pin_memory=False,  # Reduce memory usage
)

print("Training configuration:")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Output directory: {training_args.output_dir}")

## Start Fine-tuning

Now let's start the fine-tuning process!

In [ ]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_final,
    eval_dataset=eval_dataset_final,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("🚀 Starting fine-tuning...")
print(f"Training on {len(train_dataset_final)} examples")
print(f"Validating on {len(eval_dataset_final)} examples")

# Start training
start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time

print(f"\n✅ Training completed in {training_time:.2f} seconds")
print(f"Training loss: {train_result.training_loss:.4f}")

# Save the model
trainer.save_model()
print("Model saved to ./fine_tuned_model")

## Evaluate Fine-tuned Model

Let's test our fine-tuned model and compare it with the base model.

In [ ]:
def generate_response(model, tokenizer, prompt, max_length=100):
    """Generate response from model"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):].strip()

# Test prompts
test_prompts = [
    "### Instruction:\nWrite a Python function to reverse a string\n\n### Response:\n",
    "### Instruction:\nExplain what deep learning is\n\n### Response:\n",
    "### Instruction:\nWrite a SQL query to count records in a table\n\n### Response:\n"
]

print("🧪 Testing fine-tuned model:")
print("=" * 60)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\nTest {i}:")
    print(f"Prompt: {prompt.split('### Response:')[0].strip()}")
    
    response = generate_response(model, tokenizer, prompt, max_length=150)
    print(f"Response: {response}")
    print("-" * 40)

## Compare Base vs Fine-tuned Model

Let's load the base model and compare responses.

In [ ]:
# Load base model for comparison
print("Loading base model for comparison...")

# Clear current model from memory
del model
torch.cuda.empty_cache()

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

print("Base model loaded for comparison")

# Compare responses
comparison_prompt = "### Instruction:\nWrite a Python function to calculate the factorial of a number\n\n### Response:\n"

print("\n📊 Model Comparison:")
print("=" * 60)
print(f"Prompt: {comparison_prompt.split('### Response:')[0].strip()}")

# Base model response
base_response = generate_response(base_model, tokenizer, comparison_prompt, max_length=150)
print(f"\nBase Model Response:\n{base_response}")

# Load fine-tuned model
print("\nLoading fine-tuned model...")
del base_model
torch.cuda.empty_cache()

# Load the fine-tuned model
fine_tuned_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

# Load LoRA weights
fine_tuned_model = PeftModel.from_pretrained(fine_tuned_model, "./fine_tuned_model")

# Fine-tuned model response
fine_tuned_response = generate_response(fine_tuned_model, tokenizer, comparison_prompt, max_length=150)
print(f"\nFine-tuned Model Response:\n{fine_tuned_response}")

## Multi-GPU Training (Optional)

If you have access to multiple GPUs, here's how to set up distributed training.

In [ ]:
def setup_distributed_training():
    """Setup for distributed training across multiple GPUs"""
    
    if torch.cuda.device_count() > 1:
        print(f"🎯 Found {torch.cuda.device_count()} GPUs")
        
        # Distributed training arguments
        distributed_args = TrainingArguments(
            output_dir="./distributed_fine_tuned_model",
            num_train_epochs=2,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            warmup_steps=10,
            learning_rate=2e-4,
            fp16=True,
            logging_steps=1,
            evaluation_strategy="steps",
            eval_steps=5,
            save_steps=10,
            remove_unused_columns=False,
            dataloader_pin_memory=False,
            # Distributed training settings
            ddp_find_unused_parameters=False,
            dataloader_num_workers=0,
        )
        
        print("Distributed training configuration:")
        print(f"Batch size per GPU: {distributed_args.per_device_train_batch_size}")
        print(f"Total effective batch size: {distributed_args.per_device_train_batch_size * torch.cuda.device_count()}")
        
        return distributed_args
    else:
        print("❌ Only 1 GPU available. Distributed training requires multiple GPUs.")
        return None

# Check for multi-GPU setup
distributed_args = setup_distributed_training()

if distributed_args:
    print("\n💡 To run distributed training:")
    print("1. Use torchrun or accelerate launch")
    print("2. Set CUDA_VISIBLE_DEVICES appropriately")
    print("3. Ensure all GPUs have sufficient memory")
else:
    print("\n💡 Single GPU training is sufficient for this workshop.")

## Performance Analysis

Let's analyze the training performance and memory usage.

In [ ]:
def analyze_training_performance():
    """Analyze training performance metrics"""
    
    print("📊 Training Performance Analysis:")
    print("=" * 50)
    
    # Memory usage
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        
        print(f"GPU Memory Usage:")
        print(f"  Allocated: {allocated:.2f} GB")
        print(f"  Reserved: {reserved:.2f} GB")
        print(f"  Total Available: {total:.2f} GB")
        print(f"  Utilization: {(allocated/total)*100:.1f}%")
    
    # Training efficiency
    print(f"\nTraining Efficiency:")
    print(f"  Examples per epoch: {len(train_dataset_final)}")
    print(f"  Batch size: {training_args.per_device_train_batch_size}")
    print(f"  Steps per epoch: {len(train_dataset_final) // training_args.per_device_train_batch_size}")
    print(f"  Total training steps: {training_args.num_train_epochs * (len(train_dataset_final) // training_args.per_device_train_batch_size)}")
    
    # Model size analysis
    total_params = sum(p.numel() for p in fine_tuned_model.parameters())
    trainable_params = sum(p.numel() for p in fine_tuned_model.parameters() if p.requires_grad)
    
    print(f"\nModel Size Analysis:")
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Trainable percentage: {(trainable_params/total_params)*100:.2f}%")
    
    # LoRA efficiency
    print(f"\nLoRA Efficiency:")
    print(f"  Rank: {lora_config.r}")
    print(f"  Alpha: {lora_config.lora_alpha}")
    print(f"  Memory reduction: ~{(1 - trainable_params/total_params)*100:.1f}%")

analyze_training_performance()

## Save Model for Deployment

Let's save our fine-tuned model in a format suitable for deployment.

In [ ]:
def save_model_for_deployment():
    """Save model in deployment-ready format"""
    
    # Create deployment directory
    deployment_dir = Path("./deployment_model")
    deployment_dir.mkdir(exist_ok=True)
    
    # Save LoRA adapter
    fine_tuned_model.save_pretrained(str(deployment_dir / "lora_adapter"))
    
    # Save tokenizer
    tokenizer.save_pretrained(str(deployment_dir / "tokenizer"))
    
    # Save configuration
    config = {
        "base_model": MODEL_NAME,
        "lora_config": {
            "r": lora_config.r,
            "alpha": lora_config.lora_alpha,
            "target_modules": lora_config.target_modules,
            "dropout": lora_config.lora_dropout
        },
        "training_args": {
            "learning_rate": training_args.learning_rate,
            "num_epochs": training_args.num_train_epochs,
            "batch_size": training_args.per_device_train_batch_size
        }
    }
    
    with open(deployment_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    
    # Create deployment script
    deployment_script = """
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import json

def load_fine_tuned_model(model_path="./deployment_model"):
    # Load configuration
    with open(f"{model_path}/config.json", "r") as f:
        config = json.load(f)
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(f"{model_path}/tokenizer")
    
    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        config["base_model"],
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    # Load LoRA adapter
    model = PeftModel.from_pretrained(base_model, f"{model_path}/lora_adapter")
    
    return model, tokenizer

if __name__ == "__main__":
    model, tokenizer = load_fine_tuned_model()
    print("Model loaded successfully!")
"""
    
    with open(deployment_dir / "load_model.py", "w") as f:
        f.write(deployment_script)
    
    print(f"✅ Model saved for deployment in: {deployment_dir}")
    print(f"Files created:")
    print(f"  - lora_adapter/ (LoRA weights)")
    print(f"  - tokenizer/ (Tokenizer)")
    print(f"  - config.json (Configuration)")
    print(f"  - load_model.py (Loading script)")

save_model_for_deployment()

## Cleanup

Let's clean up GPU memory and summarize what we've accomplished.

In [ ]:
def final_cleanup():
    """Final cleanup and summary"""
    
    # Clean up models
    if 'fine_tuned_model' in globals():
        del fine_tuned_model
    if 'tokenizer' in globals():
        del tokenizer
    
    # Clear GPU memory
    torch.cuda.empty_cache()
    
    print("🧹 Cleanup completed")
    
    # Summary
    print("\n📋 Fine-tuning Summary:")
    print("=" * 40)
    print(f"✅ Base model: {MODEL_NAME}")
    print(f"✅ Training examples: {len(train_dataset_final)}")
    print(f"✅ Validation examples: {len(eval_dataset_final)}")
    print(f"✅ Training epochs: {training_args.num_train_epochs}")
    print(f"✅ LoRA rank: {lora_config.r}")
    print(f"✅ Model saved to: ./fine_tuned_model")
    print(f"✅ Deployment package: ./deployment_model")
    
    print("\n🎯 Next steps:")
    print("1. Test your fine-tuned model with new prompts")
    print("2. Deploy using the deployment package")
    print("3. Experiment with different LoRA configurations")
    print("4. Try fine-tuning on larger datasets")

# Uncomment to run cleanup
# final_cleanup()

print("💡 Uncomment final_cleanup() to clean up GPU memory")

## Exercises

### Basic Exercises
1. **Dataset Creation**: Create your own instruction dataset for:
   - Code review and suggestions
   - Creative writing prompts
   - Technical documentation

2. **LoRA Configuration**: Experiment with different LoRA settings:
   - Rank (r): 8, 16, 32, 64
   - Alpha: 16, 32, 64
   - Target modules: Try different combinations

3. **Training Parameters**: Optimize training settings:
   - Learning rates: 1e-5, 2e-4, 5e-4
   - Batch sizes: 1, 2, 4
   - Epochs: 1, 3, 5

### Advanced Exercises
4. **Model Comparison**: Fine-tune different base models:
   - Phi-2 vs Mistral-7B
   - Compare performance and memory usage
   - Evaluate on specific tasks

5. **QLoRA Implementation**: Implement 4-bit quantization:
   - Use BitsAndBytesConfig with 4-bit
   - Compare memory usage vs 8-bit
   - Measure performance impact

6. **Evaluation Metrics**: Implement comprehensive evaluation:
   - Perplexity calculation
   - BLEU scores for text generation
   - Human evaluation metrics

7. **Distributed Training**: Set up multi-GPU training:
   - Use accelerate or torchrun
   - Optimize batch sizes across GPUs
   - Monitor training efficiency

---
**Next notebook:** [Production Deployment with Slurm](04_slurm_production.ipynb)

// ...at the top of each notebook...
{
 "cell_type": "markdown",
 "metadata": {},
 "source": [
  "© mattbixley"
 ]
},